# Near Miss Detection

In [ ]:
# Imports
import pandas as pd
import numpy as np 
import pathlib
import time
import os
from datetime import datetime, timedelta

In [ ]:
# Old
START_DATE = "2025-06-02"
END_DATE = "2025-06-02"

DATA_DIR = "C:/Users/suggu/IITM/AGC/flow-analytics/Data/2025-06-02-data/2nd_June_2025"

def load_data(data_dir, start_date, end_date):
    
    # List to collect dataframes
    dfs = []
    
    # Loop over all folders within date range
    for folder in sorted(os.listdir(data_dir)):
        folder_path = os.path.join(data_dir, folder)
        
        # Skip non-folders
        if not os.path.isdir(folder_path):
            continue
        
        if folder.startswith(START_DATE) or folder.startswith(END_DATE):
            # Load all parquet files inside the folder
            dfs.append(pd.read_parquet(folder_path))
    
    # Combine all into single dataframe
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

# Usage
df = load_data(DATA_DIR, START_DATE, END_DATE)
print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

Loaded 48283533 records from 2025-06-02 to 2025-06-02


In [ ]:
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-02 00:00:00.041107893,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.236328,-0.0,0.017080
1,2025-06-02 00:00:00.140780926,8748438,4,61.93750,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.244141,0.0,0.018615
2,2025-06-02 00:00:00.249500036,8748438,4,61.93750,-58.50000,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.242188,-0.0,0.025484
3,2025-06-02 00:00:00.338927984,8748438,4,61.93750,-58.46875,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.238281,-0.0,0.006648
4,2025-06-02 00:00:00.436949015,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.236328,-0.0,0.027398


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48283533 entries, 0 to 48283532
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


In [ ]:
len(df['id'].unique()) , len(df['label'].unique())

(86426, 8)

# Lifetime Filtering

In [ ]:
# Global minimum detection count required per label
global_lifespan_thresholds = {
        1: 30,
        2: 80,
        3: 60,
        4: 90,
        5: 30,
        6: 100,
        7: 100,   
        8: 180
    }

# Compute lifespan as detection count in full dataset
lifespan = (
    df.groupby(["id", "label"])["timestamp"]
    .count()
    .reset_index(name="lifespan")
)


# Attach thresholds
lifespan["min_required"] = lifespan["label"].map(global_lifespan_thresholds)

# Identify short-lived objects
lifespan["is_outlier"] = lifespan["lifespan"] < lifespan["min_required"]

# Get outlier IDs
short_lived_ids = set(lifespan.loc[lifespan["is_outlier"], "id"].tolist())

# Drop them from the cleaned dataframe
df = df[~df["id"].isin(short_lived_ids)]

# Logging
print(f"[lifespan filter] Removed {len(short_lived_ids)} short-lived IDs")

[lifespan filter] Removed 5651 short-lived IDs


- pedestrian=1
- bicycle=2
- motorcycle=3
- car=4
- escooter=5,
- van=6
- truck=7
- bus=8

# foothpath zones filtering

In [66]:
footpath_zones = [{"id":"1081","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((-8.6426068 4.1825497, -5.2591289 6.6106291, 2.4873278 -1.8810115, 3.0701297 -4.1885040, -4.8739637 -7.3121410, -5.9190415 -5.1056689, -14.9469623 -8.3614314, -15.4947729 -6.7531838, -6.4707399 -3.4197283, -5.5463181 -0.1021501, -8.6426068 4.1825497))","min_z":-1.5,"max_z":3.5},
{"id":"1082","name":"FalseDetection (Vehicle as Pedestrian)","type":"analytics","vertices":"POLYGON ((-3.3894790 -13.3168241, 11.5404031 -7.8269771, 18.9154074 -17.2223685, 14.9261910 -19.9865761, 21.4404136 -27.6760383, 20.0765345 -28.6872835, 7.1069287 -14.4841637, -11.9112838 -20.7401988, -12.4875105 -18.6473010, -2.5169133 -15.3193791, -3.3894790 -13.3168241))","min_z":-1.5,"max_z":3.5},
{"id":"1083","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((65.0702212 14.5604360, 36.5624449 3.1898636, 35.8594607 -0.3697733, 48.5353798 -17.2319024, 52.8105665 -14.6263596, 49.1754262 -9.8991876, 52.2197357 2.2111523, 68.4919414 9.0673664, 65.0702212 14.5604360))","min_z":-1.5,"max_z":3.5},
{"id":"1084","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((5.9894856 29.2443364, 7.5000886 30.3643575, 15.4508444 22.7984588, 22.8276900 37.1368332, 27.9001775 34.9101554, 20.8424398 19.4127592, 17.0919999 16.0917894, 5.9894856 29.2443364))","min_z":-1.5,"max_z":3.5}]

import geopandas as gpd
from shapely import wkt

zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")
columns = df.columns.tolist()

In [68]:
def attach_zones_to_objects(df, gdf_zones, how = "inner", batch_size=100000):

    num_chunks = len(df) // batch_size + 1
    
    output_chunks = []

    for i in range(num_chunks):
        chunk = df.iloc[i*batch_size : (i+1)*batch_size].copy()

        if len(chunk) == 0:
            continue

        gdf_chunk = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["pos_x"], chunk["pos_y"]),
        )

        joined = gpd.sjoin(gdf_chunk, gdf_zones, how=how, predicate="within")

        # Rename columns to meaningful names
        joined = joined.rename(columns={
            "id_left": "id",
            "id_right": "zone"
        })

        joined = joined[columns + ["zone"]]

        output_chunks.append(joined)

    return pd.concat(output_chunks, ignore_index=True)


In [69]:
df = attach_zones_to_objects(df, gdf_zones, how = "left")

In [ ]:
def apply_footpath_zone_filter(df):

    max_speed = {
        1: 4.0,
        2: 6.0,
        3: 12.0,
        4: 12.0,
        5: 4.0,
        6: 12.0,
        7: 0.0,
        8: 0.0,
    }

    df_zone = df[df["zone"].notnull()].copy()
    speed_limit_series = df_zone["label"].map(max_speed)

    forbidden_mask = df_zone["label"].isin(
        [3, 4, 6, 7, 8]
    )

    speed_exceed_mask  = df_zone["vel"] > speed_limit_series

    remove_mask = forbidden_mask | speed_exceed_mask

    removed_ids = df_zone.loc[remove_mask, "id"].unique()
    df = df.loc[~df["id"].isin(removed_ids)].copy()

    print(f"[footpath zone] Removed {len(removed_ids)} objects")

    return df

In [71]:
df = apply_footpath_zone_filter(df)

[footpath zone] Removed 457 objects


In [72]:
df.drop(columns=["zone"], inplace=True)
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-02 00:00:00.041107893,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.236328,-0.0,0.017080
1,2025-06-02 00:00:00.140780926,8748438,4,61.93750,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.244141,0.0,0.018615
2,2025-06-02 00:00:00.249500036,8748438,4,61.93750,-58.50000,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.242188,-0.0,0.025484
3,2025-06-02 00:00:00.338927984,8748438,4,61.93750,-58.46875,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.238281,-0.0,0.006648
4,2025-06-02 00:00:00.436949015,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.236328,-0.0,0.027398


In [73]:
df.reset_index(drop=True, inplace=True)
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-02 00:00:00.041107893,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.236328,-0.0,0.017080
1,2025-06-02 00:00:00.140780926,8748438,4,61.93750,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.244141,0.0,0.018615
2,2025-06-02 00:00:00.249500036,8748438,4,61.93750,-58.50000,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.242188,-0.0,0.025484
3,2025-06-02 00:00:00.338927984,8748438,4,61.93750,-58.46875,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.238281,-0.0,0.006648
4,2025-06-02 00:00:00.436949015,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.236328,-0.0,0.027398


# Cross - Walking zones filtering

In [74]:
crosswalk_zones = [{"id":"1015","name":"Crosswalk Houba - South","type":"analytics","vertices":"POLYGON ((25.0 -23.6, 42.8 -8.5, 40.3 -5.5, 22.6 -21.0, 25.0 -23.6))","min_z":-1.5,"max_z":3.5},
{"id":"1016","name":"Crosswalk Amandiers","type":"analytics","vertices":"POLYGON ((-2.7 -13.4, -4.8 -7.4, -1.4 -6.0, 0.7 -12.2, -2.7 -13.4))","min_z":-1.5,"max_z":3.5},
{"id":"1017","name":"Crosswalk Houba - North","type":"analytics","vertices":"POLYGON ((-3.1 4.7, 13.8 19.3, 16.9 16.1, 0.0 1.4, -3.1 4.7))","min_z":-1.5,"max_z":3.5},
{"id":"1018","name":"Crosswalk Magnolias","type":"analytics","vertices":"POLYGON ((30.5 15.5, 32.2 18.9, 23.4 23.5, 21.6 20.2, 30.5 15.5))","min_z":-1.5,"max_z":3.5},
{"id":"1019","name":"Crosswalk Charlotte [1]","type":"analytics","vertices":"POLYGON ((36.4 15.3, 40.4 5.2, 37.1 3.8, 33.1 14.0, 36.4 15.3))","min_z":-1.5,"max_z":3.5}]

zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")


In [ ]:
# ---------------------------------------------
# 4. Compute polygon orientation (longest edge)
# ---------------------------------------------
def compute_polygon_orientation(poly):
    coords = list(poly.exterior.coords)
    edges = []

    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        dx, dy = x2 - x1, y2 - y1
        length = (dx**2 + dy**2) ** 0.5
        edges.append((length, dx, dy))

    # longest edge
    _, dx, dy = max(edges, key=lambda e: e[0])
    return np.degrees(np.arctan2(dy, dx))



# ---------------------------------------------
# 5. Apply zone-level filtering for cars
# ---------------------------------------------
def filter_parallel_cars(df_zone, orientation_deg, threshold=4.0, l):
    """
    df_zone contains objects inside a single crosswalk zone
    """

    cars = df_zone[df_zone["label"] == l].copy()  # label=4 = car

    if cars.empty:
        return [], df_zone  # nothing removed

    # Compute heading
    cars["heading_deg"] = np.degrees(cars["yaw"])

    # Fallback if yaw is missing
    missing = cars["heading_deg"].isna()
    cars.loc[missing, "heading_deg"] = np.degrees(
        np.arctan2(cars.loc[missing, "vel_y"], cars.loc[missing, "vel_x"])
    )

    # Angular difference
    def angle_diff(a, b):
        d = (a - b + 180) % 360 - 180
        return abs(d)

    cars["angle_diff"] = cars.apply(
        lambda r: angle_diff(r["heading_deg"], orientation_deg),
        axis=1
    )

    # Cars moving parallel to crosswalk axis
    parallel = cars[cars["angle_diff"] < threshold]["id"].unique().tolist()

    # Remove them
    df_zone_filtered = df_zone[~df_zone["id"].isin(parallel)]

    return parallel, df_zone_filtered


In [ ]:
# Precompute zone orientations
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

removed_ids_global = []

df = attach_zones_to_objects(df, gdf_zones, how = "left")

# Process each zone separately
for zone_id, zone_df in df.groupby("zone"): 
    print(f"Processing crosswalk zone {zone_id} with {len(zone_df)} objects")

    zone_orientation = float(
        gdf_zones.loc[gdf_zones["id"] == str(zone_id), "orientation_deg"].iloc[0]
    )
    for l in range(3, 8):
        removed_ids, zone_filtered = filter_parallel_cars(
            zone_df, orientation_deg=zone_orientation, threshold=4.0, l
        )

    removed_ids_global.append(removed_ids)

    print(f"[crosswalk zone] Zone {zone_id}: Removed {len(removed_ids)} parallel cars")

removed_ids_global = [item for sublist in removed_ids_global for item in sublist]
df = df[~df["id"].isin(removed_ids_global)]

Processing crosswalk zone 1015 with 371655 objects
[crosswalk zone] Zone 1015: Removed 5 parallel cars
Processing crosswalk zone 1016 with 97160 objects
[crosswalk zone] Zone 1016: Removed 2 parallel cars
Processing crosswalk zone 1017 with 468078 objects
[crosswalk zone] Zone 1017: Removed 2 parallel cars
Processing crosswalk zone 1018 with 326920 objects
[crosswalk zone] Zone 1018: Removed 0 parallel cars
Processing crosswalk zone 1019 with 269942 objects
[crosswalk zone] Zone 1019: Removed 0 parallel cars
